# Retail medallion bronze ingestion

This notebook ingests the committed retail-purchases sample files from the lakehouse landing zone.
It materializes raw purchases, customers, and products into bronze Delta tables with explicit metadata,
symbolic Fabric placeholders, and forward-compatible handling for upcoming `customer_name`, `product_name`,
and `unit_cost` columns.


In [ ]:
ENVIRONMENT = "dev"
PROCESSING_DATE = "2026-03-02"
PIPELINE_RUN_ID = "local-demo"
environment = ENVIRONMENT
processing_date = PROCESSING_DATE
pipeline_run_id = PIPELINE_RUN_ID


In [ ]:
from pyspark.sql import functions as F
import json
import uuid

batch_id = f"retail-bronze-{processing_date}-{uuid.uuid4()}"

landing_root = 'Files/landing/retail-purchases'
purchases_path = f'{landing_root}/purchases.csv'
customers_path = f'{landing_root}/customers.csv'
products_path = f'{landing_root}/products.csv'

purchase_input_columns = [
    'purchase_id', 'customer_id', 'product_id', 'purchase_ts', 'ingest_ts',
    'quantity', 'unit_price', 'store_id', 'channel', 'synthetic_record', 'anomaly_label'
]
customer_input_columns = [
    'customer_id', 'customer_name', 'loyalty_tier', 'home_region', 'signup_date', 'email_domain', 'synthetic_record'
]
product_input_columns = [
    'product_id', 'product_name', 'category', 'subcategory', 'unit_cost', 'unit_price', 'is_active', 'synthetic_record'
]

purchases_raw_table = 'bronze.retail_purchases_raw'
customers_raw_table = 'bronze.retail_customers_raw'
products_raw_table = 'bronze.retail_products_raw'
audit_table = 'bronze.retail_ingestion_audit'


In [ ]:
def log_event(stage, message, **metrics):
    record = {
        'stage': stage,
        'message': message,
        'environment': environment,
        'processing_date': processing_date,
        'pipeline_run_id': pipeline_run_id,
        'batch_id': batch_id,
        'metrics': metrics,
    }
    print(json.dumps(record, sort_keys=True))

def read_csv_with_header(path, required_columns):
    df = spark.read.format('csv').option('header', True).load(path)
    missing_columns = [column for column in required_columns if column not in df.columns]
    assert not missing_columns, f'Missing required columns for {path}: {missing_columns}'
    return df.withColumn('_source_file', F.input_file_name())

def with_bronze_metadata(df, entity_name):
    return (
        df.withColumn('metadata_ingestion_timestamp', F.current_timestamp())
        .withColumn('metadata_source_entity', F.lit(entity_name))
        .withColumn('metadata_batch_id', F.lit(batch_id))
        .withColumn('metadata_pipeline_run_id', F.lit(pipeline_run_id))
        .withColumn('metadata_processing_date', F.lit(processing_date))
        .withColumnRenamed('_source_file', 'source_file')
    )

spark.sql('CREATE SCHEMA IF NOT EXISTS bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS gold')

purchase_source_df = read_csv_with_header(purchases_path, purchase_input_columns)
customer_source_df = read_csv_with_header(customers_path, customer_input_columns)
product_source_df = read_csv_with_header(products_path, product_input_columns)


In [ ]:
purchases_raw_df = purchase_source_df.select(
    F.col('purchase_id').cast('string').alias('purchase_id'),
    F.col('customer_id').cast('string').alias('customer_id'),
    F.col('product_id').cast('string').alias('product_id'),
    F.to_timestamp('purchase_ts').alias('purchase_ts'),
    F.to_timestamp('ingest_ts').alias('ingest_ts'),
    F.col('quantity').cast('int').alias('quantity'),
    F.col('unit_price').cast('decimal(18,2)').alias('unit_price'),
    F.col('store_id').cast('string').alias('store_id'),
    F.col('channel').cast('string').alias('channel'),
    F.col('synthetic_record').cast('string').alias('synthetic_record'),
    F.col('anomaly_label').cast('string').alias('anomaly_label'),
    F.col('_source_file'),
)
purchases_raw_df = with_bronze_metadata(purchases_raw_df, 'purchases').withColumn('source_ingest_date', F.to_date('ingest_ts'))

customer_select_exprs = [
    F.col('customer_id').cast('string').alias('customer_id'),
    F.col('loyalty_tier').cast('string').alias('loyalty_tier'),
    F.col('home_region').cast('string').alias('home_region'),
    F.to_date('signup_date').alias('signup_date'),
    F.col('email_domain').cast('string').alias('email_domain'),
    F.col('synthetic_record').cast('string').alias('synthetic_record'),
    F.col('_source_file'),
]
if 'customer_name' in customer_source_df.columns:
    customer_select_exprs.append(F.col('customer_name').cast('string').alias('customer_name'))
customers_raw_df = customer_source_df.select(*customer_select_exprs)
if 'customer_name' not in customers_raw_df.columns:
    customers_raw_df = customers_raw_df.withColumn(
        'customer_name',
        F.concat(F.lit('Customer '), F.regexp_extract('customer_id', r'(\d+)$', 1)),
    )
customers_raw_df = with_bronze_metadata(customers_raw_df, 'customers')

product_select_exprs = [
    F.col('product_id').cast('string').alias('product_id'),
    F.col('category').cast('string').alias('category'),
    F.col('subcategory').cast('string').alias('subcategory'),
    F.col('unit_price').cast('decimal(18,2)').alias('unit_price'),
    F.col('is_active').cast('string').alias('is_active'),
    F.col('synthetic_record').cast('string').alias('synthetic_record'),
    F.col('_source_file'),
]
if 'product_name' in product_source_df.columns:
    product_select_exprs.append(F.col('product_name').cast('string').alias('product_name'))
if 'unit_cost' in product_source_df.columns:
    product_select_exprs.append(F.col('unit_cost').cast('decimal(18,2)').alias('unit_cost'))
products_raw_df = product_source_df.select(*product_select_exprs)
if 'product_name' not in products_raw_df.columns:
    products_raw_df = products_raw_df.withColumn(
        'product_name',
        F.concat(F.lit('Product '), F.regexp_extract('product_id', r'(\d+)$', 1)),
    )
if 'unit_cost' not in products_raw_df.columns:
    products_raw_df = products_raw_df.withColumn('unit_cost', F.round(F.col('unit_price') * F.lit(0.62), 2))
products_raw_df = with_bronze_metadata(products_raw_df, 'products')


In [ ]:
(
    purchases_raw_df.write.format('delta')
    .mode('overwrite')
    .partitionBy('source_ingest_date')
    .option('overwriteSchema', 'true')
    .saveAsTable(purchases_raw_table)
)
customers_raw_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(customers_raw_table)
products_raw_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(products_raw_table)

audit_df = spark.createDataFrame([
    ('purchases', purchases_path, purchases_raw_df.count()),
    ('customers', customers_path, customers_raw_df.count()),
    ('products', products_path, products_raw_df.count()),
], 'entity_name string, source_path string, row_count long').withColumn('processing_date', F.lit(processing_date))       .withColumn('pipeline_run_id', F.lit(pipeline_run_id))       .withColumn('batch_id', F.lit(batch_id))       .withColumn('logged_at', F.current_timestamp())

audit_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(audit_table)

log_event(
    'bronze_validation',
    'Retail bronze ingestion complete',
    purchases_rows=purchases_raw_df.count(),
    customers_rows=customers_raw_df.count(),
    products_rows=products_raw_df.count(),
)
spark.sql(f'SELECT COUNT(*) AS purchases_rows FROM {purchases_raw_table}').show()
spark.sql(f'SELECT COUNT(*) AS customers_rows FROM {customers_raw_table}').show()
spark.sql(f'SELECT COUNT(*) AS products_rows FROM {products_raw_table}').show()
